In [1]:
import numpy as np
import pandas as pd
from sklearn.utils import compute_class_weight
from torch.nn import CrossEntropyLoss
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from torch.utils.data import Dataset,DataLoader
import torch
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
import warnings
warnings.filterwarnings("ignore")

/home/acer/ml_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_train=pd.read_csv("/mnt/d/EmoBase/Data/train_sent_emo.csv")
df_dev=pd.read_csv("/mnt/d/EmoBase/Data/dev_sent_emo.csv")
df_test=pd.read_csv("/mnt/d/EmoBase/Data/test_sent_emo.csv")

In [3]:
df_train.head(5)

,Sr No.,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime
0,1,also I was the point person on my company’s tr...,Chandler,neutral,neutral,0,0,8,21,"00:16:16,059","00:16:21,731"
1,2,You must’ve had your hands full.,The Interviewer,neutral,neutral,0,1,8,21,"00:16:21,940","00:16:23,442"
2,3,That I did. That I did.,Chandler,neutral,neutral,0,2,8,21,"00:16:23,442","00:16:26,389"
3,4,So let’s talk a little bit about your duties.,The Interviewer,neutral,neutral,0,3,8,21,"00:16:26,820","00:16:29,572"
4,5,My duties? All right.,Chandler,surprise,positive,0,4,8,21,"00:16:34,452","00:16:40,917"


In [4]:
df_dev.isnull().sum()

Sr No.          0
Utterance       0
Speaker         0
Emotion         0
Sentiment       0
Dialogue_ID     0
Utterance_ID    0
Season          0
Episode         0
StartTime       0
EndTime         0
dtype: int64

In [5]:
df_test.isnull().sum()

Sr No.          0
Utterance       0
Speaker         0
Emotion         0
Sentiment       0
Dialogue_ID     0
Utterance_ID    0
Season          0
Episode         0
StartTime       0
EndTime         0
dtype: int64

In [6]:
df_test.isnull().sum()

Sr No.          0
Utterance       0
Speaker         0
Emotion         0
Sentiment       0
Dialogue_ID     0
Utterance_ID    0
Season          0
Episode         0
StartTime       0
EndTime         0
dtype: int64

In [7]:
df_dev.columns
df_test.columns
df_test.columns

Index(['Sr No.', 'Utterance', 'Speaker', 'Emotion', 'Sentiment', 'Dialogue_ID',
       'Utterance_ID', 'Season', 'Episode', 'StartTime', 'EndTime'],
      dtype='str')

In [8]:
df_train['Emotion'].value_counts()

Emotion
neutral     4710
joy         1743
surprise    1205
anger       1109
sadness      683
disgust      271
fear         268
Name: count, dtype: int64

In [9]:
def add_context(df):
    df = df.sort_values(['Dialogue_ID', 'Utterance_ID']).reset_index(drop=True)
    contexts = []
    history = {}

    for _,row in df.iterrows():
        dia_id = row['Dialogue_ID']
        utt = row['Utterance']

        if dia_id not in history:
            history[dia_id] = []

        history[dia_id].append(utt)
        last_3 = history[dia_id][-3:]
        context = " [SEP] ".join(last_3)
        contexts.append(context)

    df['context'] = contexts
    return df

df_train = add_context(df_train)
df_dev = add_context(df_dev)
df_test = add_context(df_test)

In [10]:
print(len(df_train))
print(len(df_dev))
print(len(df_test))

9989
1109
2610


In [11]:
df_train["Utterance"] = df_train["Utterance"].str.replace("NBSP", " ")
df_dev["Utterance"] = df_dev["Utterance"].str.replace("NBSP", " ")
df_test["Utterance"] = df_test["Utterance"].str.replace("NBSP", " ")

In [12]:
label_map={
    "neutral":0,
    "joy":1,
    "sadness":2,
    "anger":3,
    "fear":4,
    "surprise":5,
    "disgust":6,
}
df_dev["label"]=df_dev["Emotion"].map(label_map)
df_test['label']=df_test["Emotion"].map(label_map)
df_train['label']=df_train["Emotion"].map(label_map)

In [13]:
df_train["label"] = df_train["label"].astype(int)
df_dev["label"] = df_dev["label"].astype(int)
df_test["label"] = df_test["label"].astype(int)

In [14]:
df_train[['Utterance', 'Emotion', 'label']].head()

,Utterance,Emotion,label
0,also I was the point person on my company’s tr...,neutral,0
1,You must’ve had your hands full.,neutral,0
2,That I did. That I did.,neutral,0
3,So let’s talk a little bit about your duties.,neutral,0
4,My duties? All right.,surprise,5


In [15]:
train_labels = torch.tensor(df_train["label"].tolist())
dev_labels = torch.tensor(df_dev["label"].tolist())
test_labels = torch.tensor(df_test["label"].tolist())

In [16]:
model=AutoModelForSequenceClassification.from_pretrained("roberta-base",num_labels=7)
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

encoded_train = tokenizer(
    df_train["Utterance"].tolist(),
    max_length=128,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)


encoded_dev=tokenizer(df_dev["Utterance"].tolist(),max_length=128,return_tensors="pt",padding="max_length",truncation=True)

encoded_test=tokenizer(df_test["Utterance"].tolist(),max_length=128,return_tensors="pt",padding="max_length",truncation=True)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2754.60it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
class Emo(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "labels": torch.tensor(self.labels[idx])
        }
        return item

In [18]:
train_dataset=Emo(encoded_train,train_labels)
dev_dataset=Emo(encoded_dev,dev_labels)
test_dataset = Emo(encoded_test, test_labels)

In [ ]:
test_loader=DataLoader(test_dataset,batch_size=16,shuffle=False)
train_loader=DataLoader(train_dataset,batch_size=16,shuffle=True)
dev_loader=DataLoader(dev_dataset,batch_size=16,shuffle=False)

y=train_labels.numpy()
class_weight=compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2,3,4,5,6]),
    y=y
)

In [20]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
model = model.to(device)

Using device: cuda


In [21]:
print(torch.cuda.is_available())

True


In [ ]:
total_steps = len(train_loader) * 5
warmup_steps = total_steps // 10

optimizer=AdamW(model.parameters(),lr=1e-5)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
class_weight_tensor = torch.tensor(class_weight, dtype=torch.float).to(device)
loss_fn=CrossEntropyLoss(weight=class_weight_tensor)

for epoch in range(12):
    print(f"Epoch {epoch+1}/12")
    model.train()
    total_loss=0

    for batch in train_loader:
        input_ids=batch["input_ids"].to(device)
        attention_mask=batch["attention_mask"].to(device)
        labels=batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits=outputs.logits
        loss=loss_fn(logits,labels)

        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()


avg_loss = total_loss / len(train_loader)
print(f"Average Loss: {avg_loss:.4f}")


Epoch 1/12
Epoch 2/12
Epoch 3/12
Epoch 4/12
Epoch 5/12


In [ ]:
model.save_pretrained("emo_model")
tokenizer.save_pretrained("emo_model")

In [ ]:
model.eval()
all_pred=[]
all_label=[]

with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids,attention_mask=attention_mask)
        pred=outputs.logits.argmax(dim=1)
        all_pred.extend(pred.cpu().numpy())
        all_label.extend(labels.cpu().numpy())

from sklearn.metrics import f1_score
f1 = f1_score(all_label, all_pred, average='macro')
print(f"Macro F1: {f1:.4f}")